# Chen 2020 — осесимметричная LTE-дуга + ванна (I = 95 A)

Уровень модели ближе к статье Chen / Murphy **Fig. 7**: в домене есть **плазма Ar (LTE)**, проволока и ванна; свободная поверхность — **Kim–Na** (равновесие энергии), не VOF.

**Включено**
- Осесимметрия \((r,z)\): пластина + столб плазмы + кончик проволоки
- Ток / \(\mathbf{j}\times\mathbf{B}\), джоуль, теплопроводность, упрощённый NEC Ar
- Sheath-нагрев (Lowke–Tanaka, упрощённо)
- Усреднённая капля (масса / импульс / энтальпия)
- Поверхность \(\varphi(r)\) с объёмом наплавки \(V=A\cdot L_\mathrm{char}\)

**Не включено (отличия от ArcWeld)**
- 3D система горелки, пары Al в плазме, полный GUI
- Совпадение с Table 2 **не гарантируется** (свойства Ar — открытые таблицы)

Плоская 3D-ванна без плазмы остаётся в `weldpool_chen.py` как baseline.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

from weldpool_chen_axi import WeldPoolChenAxi, MetalProps, bop_95A_axi, quick_bop_95A_axi

ROOT = Path('.').resolve()
RESULTS = ROOT / 'results_axi_95A.npz'

TABLE2 = {
    'measured': {'RH_mm': 1.47, 'WD_mm': 1.85, 'BW_mm': 5.92, 'gamma': 0.62},
    'chen_paper': {'RH_mm': 1.54, 'WD_mm': 1.76, 'BW_mm': 5.54, 'gamma': 0.59},
}

QUICK = False
RUN_IF_MISSING = True

In [ ]:
def run_and_save(quick=False):
    proc = quick_bop_95A_axi() if quick else bop_95A_axi()
    model = WeldPoolChenAxi(metal=MetalProps(), process=proc)
    print(f'I={proc.I_arc} A, grid {proc.nr}x{proc.nz}, n_outer={proc.n_outer}')
    model.run(progress=True)
    s = model.summary()
    if not quick:
        np.savez_compressed(
            RESULTS,
            rc=model.rc, zc=model.zc, T=model.T, fL=model.fL,
            u=0.5*(model.u[:-1,:]+model.u[1:,:]),
            w=0.5*(model.w[:,:-1]+model.w[:,1:]),
            Jr=model.Jr, Jz=model.Jz, Bth=model.Bth, pot=model.pot,
            phi_surf=model.phi_surf,
            metal=model.metal, plasma=model.plasma, wire=model.wire,
            Fr=model.Fr, Fz=model.Fz,
            summary_json=json.dumps(s),
            I_arc=model.proc.I_arc, H_plate=model.proc.H_plate,
            z_wire_tip=model.proc.z_wire_tip,
        )
        print('saved', RESULTS)
    return model, s

if RESULTS.exists() and not QUICK:
    data = np.load(RESULTS, allow_pickle=True)
    summary = json.loads(str(data['summary_json']))
    print('loaded', RESULTS)
    print(summary)
    model = None
elif RUN_IF_MISSING or QUICK:
    model, summary = run_and_save(quick=QUICK)
    data = np.load(RESULTS, allow_pickle=True) if not QUICK else None
else:
    raise FileNotFoundError(RESULTS)

## Сравнение с Table 2

In [ ]:
print(f"{'Source':20s}  {'RH':>7s}  {'WD':>7s}  {'BW':>7s}  {'gamma':>7s}")
for name, r in [
    ('Measured', TABLE2['measured']),
    ('ArcWeld paper', TABLE2['chen_paper']),
    ('This LTE axi', {
        'RH_mm': summary['RH_mm'], 'WD_mm': summary['WD_mm'],
        'BW_mm': summary['BW_mm'], 'gamma': summary['gamma'],
    }),
]:
    print(f"{name:20s}  {r['RH_mm']:7.2f}  {r['WD_mm']:7.2f}  {r['BW_mm']:7.2f}  {r['gamma']:7.2f}")

## Поля (как Fig. 7) и профиль (как Fig. 5)

Пересборка рисунков: `python make_figs_axi_95A.py`

In [ ]:
import make_figs_axi_95A as figs
if RESULTS.exists():
    figs.main()
    for name in ('fig_chen_axi_95A_fields.png', 'fig_chen_axi_95A_profile.png'):
        img = plt.imread(ROOT / name)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(img); ax.axis('off'); ax.set_title(name)
        plt.show()

## Замечания

- Без паров Al дуга в демо обычно **горячее**, чем в Fig. 7 статьи; напряжение занижено на грубой сетке.
- Осесимметрия не даёт удлинённый шов вдоль скорости сварки — BW/RH сравнимы с Table 2 лишь ориентировочно.
- `QUICK = True` — короткий прогон для проверки цикла.